In [1]:
import numpy as np
import pandas as pd

In [5]:
def higham_nearest_psd_corr(C: np.ndarray, max_iter: int = 100, tol: float = 1e-12) -> np.ndarray:
    # Force symmetry
    C = 0.5 * (C + C.T)

    # Target iaddgonal = 1 for correlation
    diag_target = np.ones(C.shape[0])

    # Initialize working matrices
    Y = C.copy()
    delta_S = np.zeros_like(C)

    # Alternating projections
    for _ in range(max_iter):
        # Remove last correction
        R = Y - delta_S

        # Project onto PSD cone via eigenvalue clipping at 0
        w, V = np.linalg.eigh(0.5 * (R + R.T))
        w = np.maximum(w, 0.0)
        X = V @ np.diag(w) @ V.T
        X = 0.5 * (X + X.T)

        # Update correction term
        delta_S = X - R

        # Project onto "Fixed diagonal" set
        Y_new = X.copy()
        np.fill_diagonal(Y_new, diag_target)

        # Stop if converged
        if np.linalg.norm(Y_new - Y, ord="fro") / (np.linalg.norm(Y, ord="fro") + 1e-30) < tol:
            Y = Y_new
            break

        Y = Y_new

    return 0.5 * (Y + Y.T)


def higham_fix_cov(cov: np.ndarray) -> np.ndarray:
    cov = 0.5 * (cov + cov.T)

    # Extract standard deviation
    std = np.sqrt(np.diag(cov))
    inverse_std = np.where(std > 0, 1.0 / std, 0.0)

    # Convert covariance to correlation, then force symmetry
    corr = cov * np.outer(inverse_std, inverse_std)
    corr = 0.5 * (corr + corr.T)

    # Higham fix on correlation
    corr_psd = higham_nearest_psd_corr(corr)

    # Re-normalize diagonal to 1
    diagonal = np.sqrt(np.diag(corr_psd))
    inverse_diagonal = np.where(diagonal > 0, 1.0 / diagonal, 0.0)
    corr_psd = corr_psd * np.outer(inverse_diagonal, inverse_diagonal)

    # Convert correlation to covariance using original std
    cov_psd = corr_psd * np.outer(std, std)
    return 0.5 * (cov_psd + cov_psd.T)


np.random.seed(0)
cov_input = pd.read_csv("/Users/fuyuxuan/Downloads/test5_3.csv")

# Convert it to numpy array
cov_matrix = cov_input.values

# Apply the function fix to make covariance PSD
cov_fixed = higham_fix_cov(cov_matrix)

# Zero mean vector
mu = np.zeros(cov_fixed.shape[0])

# Simulate 100000 samples
X = np.random.multivariate_normal(mu, cov_fixed, size=100000)

# Compute output covariance matrix
cov_output = np.cov(X, rowvar=False, ddof=0)

# Convert it to DataFrame
cov_output_df = pd.DataFrame(cov_output, columns=cov_input.columns, index=cov_input.columns)

print(cov_output_df)

          x1        x2        x3        x4        x5
x1  0.084885  0.013040  0.038804  0.008272  0.003550
x2  0.013040  0.159457  0.053155  0.011270  0.004866
x3  0.038804  0.053155  0.037339  0.006204  0.002666
x4  0.008272  0.011270  0.006204  0.001685  0.000568
x5  0.003550  0.004866  0.002666  0.000568  0.000313
